# Update a Sigma connection writeback location

This notebook resolves a Sigma connection by exact name and updates its Databricks writeback catalog and schema. Sigma API credentials are read from a Databricks secret scope. The supported connection update API uses `PUT`, so the `CONNECTION_PAYLOAD` cell must contain all existing connection properties that need to be preserved.

Run with `dry_run=true` first and review the generated payload before changing it to `false`.

In [ ]:
# Databricks widgets
dbutils.widgets.text("connection_name", "", "Sigma connection name")
dbutils.widgets.text("catalog", "", "New writeback catalog")
dbutils.widgets.text("schema", "", "New writeback schema")
dbutils.widgets.text("secret_scope", "sigma", "Databricks secret scope")
dbutils.widgets.text("client_id_secret_key", "sigma_client_id", "Client ID secret key")
dbutils.widgets.text("client_secret_secret_key", "sigma_client_secret", "Client secret key")
dbutils.widgets.text("databricks_oauth_client_id_secret_key", "databricks_oauth_client_id", "Databricks OAuth client ID key")
dbutils.widgets.text("databricks_oauth_client_secret_secret_key", "databricks_oauth_client_secret", "Databricks OAuth client secret key")
dbutils.widgets.text("sigma_base_url", "https://aws-api.sigmacomputing.com", "Sigma API base URL")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run")

In [ ]:
CONNECTION_NAME = dbutils.widgets.get("connection_name").strip()
WRITEBACK_CATALOG = dbutils.widgets.get("catalog").strip()
WRITEBACK_SCHEMA = dbutils.widgets.get("schema").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
CLIENT_ID_SECRET_KEY = dbutils.widgets.get("client_id_secret_key").strip()
CLIENT_SECRET_SECRET_KEY = dbutils.widgets.get("client_secret_secret_key").strip()
DATABRICKS_OAUTH_CLIENT_ID_SECRET_KEY = dbutils.widgets.get("databricks_oauth_client_id_secret_key").strip()
DATABRICKS_OAUTH_CLIENT_SECRET_SECRET_KEY = dbutils.widgets.get("databricks_oauth_client_secret_secret_key").strip()
SIGMA_BASE_URL = dbutils.widgets.get("sigma_base_url").strip().rstrip("/")
DRY_RUN = dbutils.widgets.get("dry_run").lower() == "true"

if not CONNECTION_NAME or not WRITEBACK_CATALOG or not WRITEBACK_SCHEMA:
    raise ValueError("connection_name, catalog, and schema widgets are required.")

print(f"Connection: {CONNECTION_NAME!r}")
print(f"Writeback: {WRITEBACK_CATALOG}.{WRITEBACK_SCHEMA}")
print(f"Dry run: {DRY_RUN}")

## Existing connection payload

Replace the host, warehouse, and metadata URL placeholders. Keep all properties from the existing connection that Sigma must preserve. Sigma and Databricks OAuth credentials are loaded from the configured secret scope.

In [ ]:
CONNECTION_PAYLOAD = {
    "name": CONNECTION_NAME,
    "details": {
        "type": "databricks",
        "host": "YOUR_DATABRICKS_HOST",
        "endpoint": "YOUR_SQL_WAREHOUSE_ID",
        "useOauth": True,
        "useOrgOauth": False,
        "oauth": {
            "provider": "databricks",
            "clientId": dbutils.secrets.get(SECRET_SCOPE, DATABRICKS_OAUTH_CLIENT_ID_SECRET_KEY),
            "clientSecret": {
                "type": "plain",
                "value": dbutils.secrets.get(SECRET_SCOPE, DATABRICKS_OAUTH_CLIENT_SECRET_SECRET_KEY),
            },
            "redirectUri": f"{SIGMA_BASE_URL}/v2/oauth/1/authcode",
            "metadataUrl": "YOUR_DATABRICKS_OIDC_METADATA_URL",
            "scopes": ["openid", "profile", "email", "all-apis", "offline_access"],
        },
        "writebackSchemas": [
            {"writeCatalog": "CURRENT_CATALOG", "writeSchema": "CURRENT_SCHEMA"}
        ],
    },
    "runAsServiceAccount": False,
}

In [ ]:
import json
from copy import deepcopy
from typing import Any

import requests


def get_access_token() -> str:
    client_id = dbutils.secrets.get(SECRET_SCOPE, CLIENT_ID_SECRET_KEY)
    client_secret = dbutils.secrets.get(SECRET_SCOPE, CLIENT_SECRET_SECRET_KEY)
    response = requests.post(
        f"{SIGMA_BASE_URL}/v2/auth/token",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    token = data.get("access_token") or data.get("accessToken")
    if not token:
        raise RuntimeError("Sigma authentication response did not include an access token.")
    return token


def resolve_connection_id(token: str, connection_name: str) -> str:
    matches = []
    page = None
    headers = {"Authorization": f"Bearer {token}"}
    while True:
        params: dict[str, Any] = {"limit": 1000, "search": connection_name}
        if page:
            params["page"] = page
        response = requests.get(
            f"{SIGMA_BASE_URL}/v2/connections", headers=headers, params=params, timeout=30
        )
        response.raise_for_status()
        data = response.json()
        matches.extend(
            connection
            for connection in data.get("entries", [])
            if connection.get("name") == connection_name
            and not connection.get("isArchived", False)
        )
        page = data.get("nextPage")
        if not page:
            break

    if not matches:
        raise ValueError(f"No active Sigma connection found with name {connection_name!r}.")
    if len(matches) > 1:
        raise ValueError(f"Multiple active connections are named {connection_name!r}.")
    connection_id = matches[0].get("connectionId") or matches[0].get("id")
    if not connection_id:
        raise RuntimeError("Matched connection did not include a connection ID.")
    return str(connection_id)


def build_updated_payload(payload: dict[str, Any]) -> dict[str, Any]:
    updated = deepcopy(payload)
    details = updated.get("details")
    if not isinstance(details, dict) or details.get("type", "").lower() != "databricks":
        raise ValueError("CONNECTION_PAYLOAD must contain Databricks connection details.")
    schemas = details.setdefault("writebackSchemas", [])
    if not isinstance(schemas, list):
        raise ValueError("details.writebackSchemas must be an array.")
    location = {"writeCatalog": WRITEBACK_CATALOG, "writeSchema": WRITEBACK_SCHEMA}
    if schemas:
        if not isinstance(schemas[0], dict):
            raise ValueError("The first writebackSchemas entry must be an object.")
        schemas[0].update(location)
    else:
        schemas.append(location)
    return updated

In [ ]:
updated_payload = build_updated_payload(CONNECTION_PAYLOAD)
print(json.dumps(updated_payload, indent=2))

if DRY_RUN:
    print("[DRY RUN] No Sigma API update was submitted.")
else:
    access_token = get_access_token()
    connection_id = resolve_connection_id(access_token, CONNECTION_NAME)
    response = requests.put(
        f"{SIGMA_BASE_URL}/v2/connections/{connection_id}",
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
        json=updated_payload,
        timeout=30,
    )
    response.raise_for_status()
    result = response.json()
    print(f"[OK] Updated {CONNECTION_NAME!r} ({connection_id}).")
    print(json.dumps(result.get("writebackSchemas") or result.get("writebacks") or [], indent=2))